[Crowell et al.](https://www.biorxiv.org/content/10.1101/2025.06.23.660674v1) data, available via [Zenodo](http://10.0.20.161/zenodo.15574384)

In [ ]:
import dask
dask.config.set({"dataframe.query-planning": True})

In [ ]:
import dask.dataframe as dd

In [ ]:
import scanpy as sc
import gc
import scipy.sparse as sp

from tqdm.autonotebook import tqdm
from scipy.sparse import csr_matrix
from anndata.experimental import concat_on_disk

from cellina._spatial_utils import spatial_neighbors

In [ ]:
import sys

sys.path.append("../../scripts")

In [ ]:
import os
DATA_ROOT = os.environ.get("DATA_ROOT", ".")
base_path = os.path.join(DATA_ROOT, "datasets/crc")
#base_path = os.path.join(DATA_ROOT, "datasets/crc")
joint_adata_path = f'{base_path}/processed/crc_cosmx_wt.h5ad'

## Create merged adata

In [ ]:
data_list = ['110', '120', '210', '221', '231', '232', '242']

In [ ]:
# Create tmp files with csr matrices (since the original files have csc, which causes issues with concatenation)
tmp_files = []

for idx in tqdm(data_list, desc="Converting to CSR"):
    ad = sc.read(f"{base_path}/raw_zenodo/crc_{idx}.h5ad",
                 backup_url=f"https://zenodo.org/records/15574384/files/{idx}.h5ad?download=1")

    if sp.isspmatrix_csc(ad.X):
        ad.X = ad.X.tocsr()

    tmp = os.path.join(DATA_ROOT, f"tmp/{idx}_csr.h5ad")
    ad.write(tmp)
    tmp_files.append(tmp)

In [ ]:
# Sequentially merge the rest (avoid exploding RAM)
for idx in tqdm(data_list, desc="Merging slides"):
    concat_on_disk(
    [os.path.join(DATA_ROOT, f"tmp/{idx}_csr.h5ad") for idx in data_list],
    out_file=joint_adata_path,
    join="inner"
)

## Read Object

In [ ]:
adata = sc.read_h5ad(joint_adata_path)

run - batch identifier (1 or 2)

sec - per-slide section identifier (1 or 2)

rep - pre-run donor identifier (1-2 for 1st, 1-4 for 2nd run)

did - slide identifier (run+rep, e.g., 12 for 1st run, 2nd slide)

sid - section identifier (run+rep+sec; xx0 for single-section slides)

pid - clinical sample identifier (unique except for sections 231/232)

fov - field of view (FOV) correspondence (n/a = all FOVs)

In [ ]:
# Change back to csc - helps with downstream stuff
adata.X = adata.X.tocsc()

In [ ]:
adata.obs[[
    'rep', # 2 replicates per donor?
    'sid', # sample id
    "did", # NOTE: donor id
    # To identify concordant microenvironments across sections and domains, 
    # we further clustered cells based on the subpopulation composition of their local neighborhood to obtain 10 spatial niches
    'ctx', 
 ]].drop_duplicates()

In [ ]:
adata.obsm['spatial'] = adata.obs[['CenterX_global_px', 'CenterY_global_px']].values

In [ ]:
slide_key = 'sid'

sc.pp.filter_genes(adata, min_cells=100)

In [ ]:
adata = adata[~adata.obs['ist'].isna()]
adata.obs['ist'] = adata.obs['ist'].astype(str)

In [ ]:
adata.layers['counts'] = adata.X.copy()

In [ ]:
adata.write_h5ad(joint_adata_path)

## 2. Data Preprocessing

In [ ]:
adata = sc.read_h5ad(joint_adata_path)

In [ ]:
# Compute spatial connectivity matrix
#from benchmark_pipeline import K_NN
K_NN = 200
label_col = "ist"
slide_key = "sid"

In [ ]:
from _labels_to_coarse import LABEL_TO_COARSE

adata.obs["coarse_type"] = adata.obs[label_col].map(LABEL_TO_COARSE)
label_col = "coarse_type"

In [ ]:
sc.pp.highly_variable_genes(adata, layer='counts', flavor='seurat_v3', n_top_genes=2000, subset=True)

## Ligands

In [ ]:
import pandas as pd
import numpy as np

import requests
import io

# read csv from link
# https://github.com/ventolab/cellphonedb-data/blob/master/data/interaction_input.csv
resource = requests.get('https://raw.githubusercontent.com/ventolab/cellphonedb-data/master/data/interaction_input.csv').content
resource = io.StringIO(resource.decode('utf-8'))
resource = pd.read_csv(resource, sep=',')
# keep only PPIs
resource = resource[resource['is_ppi']][['interactors']]
# replace + with _
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('+', '_'))
# if interactors contains two '-' replace the first one with '&
resource['interactors'] = resource['interactors'].apply(lambda x: x.replace('-', '&', 1) if x.count('-') == 2 else x)
# split by - and expand
resource = resource['interactors'].str.split('-', expand=True)
# replace & with - in the first column
resource[0] = resource[0].apply(lambda x: x.replace('&', '-'))
resource.columns = ['ligand', 'receptor']


In [ ]:
# Split ligands on '_' to get list of subunits
resource['ligand'] = resource['ligand'].str.split('_')
# Explode the list into separate rows
resource = resource.explode('ligand').reset_index(drop=True)
ligands = resource['ligand'].unique()

In [ ]:
# Split receptors on '_' to get list of subunits
resource['receptor'] = resource['receptor'].str.split('_')
# Explode the list into separate rows
resource = resource.explode('receptor').reset_index(drop=True)
receptors = resource['receptor'].unique()

In [ ]:
ligands_in_data = [g for g in ligands if g in adata.var_names]
receptors_in_data = [g for g in receptors if g in adata.var_names]

In [ ]:
len(ligands_in_data), len(receptors_in_data)

In [ ]:
use_ligands = False
use_receptors = False

neighbor_features = adata.var_names
if use_ligands:
    neighbor_features = ligands_in_data
if use_receptors:
    neighbor_features = neighbor_features + receptors_in_data

In [ ]:
# 1. Find which cell types are present in ALL slides
# Count how many unique slides there are
all_slides = adata.obs[slide_key].unique()
num_slides = len(all_slides)

# 2. Count slides per cell type
slides_per_celltype = adata.obs.groupby(label_col)[slide_key].nunique()

# 3. Select cell types that appear in all slides
celltypes_in_all_slides = slides_per_celltype[slides_per_celltype == num_slides].index.tolist()

# 4. Filter obs to keep only these cell types
adata = adata[adata.obs[label_col].isin(celltypes_in_all_slides)]

In [ ]:
from cellina._spatial_utils import compute_spatial_features

In [ ]:
slides = adata.obs[slide_key].unique()
adata_out = None  # initialize output AnnData

for sid in tqdm(slides, desc="Adding spatial_x to slides"):
    ad_slide = adata[adata.obs[slide_key] == sid].copy()

    #spatial_neighbors(ad_slide, bandwidth=np.inf, max_neighbours=K_NN, standardize=False)
    spatial_neighbors(ad_slide, bandwidth=100 / 0.12028, max_neighbours=K_NN, standardize=False)
    ad_slide.obs['neighbor'] = ad_slide.obsp['spatial_connectivities'][:,0].toarray().astype(np.float32)

    sc.pp.normalize_total(ad_slide, target_sum=1e4)
    sc.pp.log1p(ad_slide)
    #ad_slide.obsm['spatial_x'] = ad_slide.obsp['spatial_connectivities'] @ ad_slide[:, neighbor_features].X / K_NN
    #ad_slide.obsm['spatial_x'] = csr_matrix(ad_slide.obsm['spatial_x']).astype(np.float32)
    compute_spatial_features(ad_slide)

    # Incremental concatenation
    if adata_out is None:
        adata_out = ad_slide
    else:
        adata_out = sc.concat([adata_out, ad_slide])

adata = adata_out

In [ ]:
adata.X.max()

In [ ]:
adata.obsm['spatial_x'].max()

In [ ]:
#adata.layers['counts'] = adata.X.copy()
adata.X = adata.layers['counts'].copy()
adata.uns = ad_slide.uns.copy()

In [ ]:
adata.obs["typ_clean"] = (
        adata.obs["typ"]
        .str.extract(r"(REF|TVA|CRC)", expand=False)
        )

In [ ]:
adata.obs["typ_clean"] = adata.obs["typ_clean"].astype(str)
adata.obs["typ"] = adata.obs["typ"].astype(str)

In [ ]:
test_samples = [110] #[221, 242]
sample_key = batch_key = "sid" # NOTE: on this or by patient, not sure
celltype_key = 'coarse_type' # NOTE: I created this
niche_key = condition_key = 'typ_clean' # NOTE: ctx - this is some cluster on cell types TBD
n_holdout = 1 #2
train_frac = 0.8

In [ ]:
adata.uns['default_params'] = {
    # Data Specific Parameters
    "dataset_name": "cosmx_crc_wt",
    "K_NN": K_NN,
    "celltype_key": celltype_key,
    "niche_key": niche_key,
    "sample_key": sample_key,
    "batch_key": batch_key,
    # Split Parameters
    "n_holdout": n_holdout,
    "n_val_samples": False,
    "test_samples": test_samples,
    "train_frac": train_frac
}

In [ ]:
adata.uns['default_params']

In [ ]:
adata

In [ ]:
adata.write_h5ad(joint_adata_path)

In [ ]:
adata = sc.read_h5ad(joint_adata_path)

In [ ]:
adata.obs_names_make_unique()

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)

In [ ]:
# Filter out cells that have nans in obs.typ_clean
adata = adata[adata.obs["typ_clean"]!='nan']

In [ ]:
adata.write_h5ad(joint_adata_path)

In [ ]:
adata

# SIMVI small adata

In [ ]:
adata = sc.read_h5ad(joint_adata_path)

In [ ]:
adata.obs.sid.value_counts()

In [ ]:
keep = [231, 242]
adata_slide = adata[adata.obs.sid.isin(keep)].copy()

In [ ]:
sc.pl.spatial(adata_slide[adata_slide.obs['sid'] == 231], color='typ_clean', spot_size=100)

In [ ]:
sc.pl.spatial(adata_slide[adata_slide.obs['sid'] == 242], color='typ_clean', spot_size=100)

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np


def subset_adata(adata):
    adatas = []
    for sid in adata.obs['sid'].unique():
        adata_slide = adata[adata.obs['sid'] == sid].copy()
        print(f"Slide {sid} has {len(adata_slide)} cells")
        coords = adata_slide.obsm['spatial']
        # deterministic center (example: geometric center)
        if sid == 242:
            center_point = coords.mean(axis=0) - [20000, 5000] # Slide 242
        else:
            center_point = coords.mean(axis=0) - [0, 15000] # Slide 231
        
        center_idx = np.argmin(((coords - center_point) ** 2).sum(axis=1))
        center = coords[center_idx].reshape(1, -1)

        nbrs = NearestNeighbors(n_neighbors=20_000).fit(coords)
        _, indices = nbrs.kneighbors(center)

        subset = adata_slide[indices[0]].copy()
        adatas.append(subset)

    # Return concatenated adata
    adata_sub = adatas[0]
    for ad in adatas[1:]:
        adata_sub = adata_sub.concatenate(ad)
    return adata_sub

adata_sub = subset_adata(adata_slide)


In [ ]:
adata_sub

In [ ]:
sc.pl.spatial(adata_sub[adata_sub.obs['sid'] == 231], color='typ_clean', spot_size=100)

In [ ]:
sc.pl.spatial(adata_sub[adata_sub.obs['sid'] == 242], color='typ_clean', spot_size=100)

In [ ]:
adata_sub.uns = adata.uns

In [ ]:
adata_sub.uns['default_params']['test_samples'] = False
adata_sub.uns['default_params']['n_holdout'] = 0
adata_sub.uns['default_params']['dataset_name'] = "crc_simvi"

In [ ]:
adata_sub.uns['default_params']

In [ ]:
adata_sub.write_h5ad(f"{base_path}/processed/crc_simvi.h5ad")